In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import urllib3

# SSL 경고 메시지 끄기
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def scrape_gongcha():
    base_url = "https://www.gong-cha.co.kr/brand/store/list?page={}&sido=&gugun=&keyword="
    store_data = []
    page = 1

    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 데이터 수집을 시작합니다.")

    while True:
        url = base_url.format(page)
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
        
        try:
            # verify=False 를 추가하여 SSL 인증 무시
            response = requests.get(url, headers=headers, verify=False)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            table_wrap = soup.select_one('.table-wrap')
            rows = table_wrap.select('.row.scroll-item')
            
            if not rows:
                print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 수집할 데이터가 더 이상 없습니다. 종료합니다.")
                break
            
            for row in rows:
                name = row.select_one('.cell.name').get_text(strip=True) if row.select_one('.cell.name') else "이름없음"
                address = row.select_one('.cell.addr').get_text(strip=True) if row.select_one('.cell.addr') else "주소없음"
                
                phone_tag = row.select_one('.cell.num')
                phone = phone_tag.get_text(strip=True) if phone_tag and phone_tag.get_text(strip=True) else "번호없음"
                
                store_data.append({"매장명": name, "주소": address, "전화번호": phone})

            print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {page}페이지 수집 완료 (현재 누적: {len(store_data)}개)")
            page += 1
            time.sleep(0.5)

        except Exception as e:
            print(f"에러 발생: {e}")
            break

    if store_data:
        df = pd.DataFrame(store_data)
        df.to_csv('gong-cha.csv', index=False, encoding='utf-8-sig')
        print(f"--- 총 {len(df)}개의 매장 정보가 'gong-cha.csv'로 저장되었습니다. ---")

if __name__ == "__main__":
    scrape_gongcha()

[2026-03-28 17:15:45] 데이터 수집을 시작합니다.
[2026-03-28 17:15:45] 1페이지 수집 완료 (현재 누적: 15개)
[2026-03-28 17:15:46] 2페이지 수집 완료 (현재 누적: 30개)
[2026-03-28 17:15:47] 3페이지 수집 완료 (현재 누적: 45개)
[2026-03-28 17:15:48] 4페이지 수집 완료 (현재 누적: 60개)
[2026-03-28 17:15:49] 5페이지 수집 완료 (현재 누적: 75개)
[2026-03-28 17:15:49] 6페이지 수집 완료 (현재 누적: 90개)
[2026-03-28 17:15:50] 7페이지 수집 완료 (현재 누적: 105개)
[2026-03-28 17:15:51] 8페이지 수집 완료 (현재 누적: 120개)
[2026-03-28 17:15:52] 9페이지 수집 완료 (현재 누적: 135개)
[2026-03-28 17:15:53] 10페이지 수집 완료 (현재 누적: 150개)
[2026-03-28 17:15:53] 11페이지 수집 완료 (현재 누적: 165개)
[2026-03-28 17:15:54] 12페이지 수집 완료 (현재 누적: 180개)
[2026-03-28 17:15:55] 13페이지 수집 완료 (현재 누적: 195개)
[2026-03-28 17:15:56] 14페이지 수집 완료 (현재 누적: 210개)
[2026-03-28 17:15:57] 15페이지 수집 완료 (현재 누적: 225개)
[2026-03-28 17:15:57] 16페이지 수집 완료 (현재 누적: 240개)
[2026-03-28 17:15:58] 17페이지 수집 완료 (현재 누적: 255개)
[2026-03-28 17:15:59] 18페이지 수집 완료 (현재 누적: 270개)
[2026-03-28 17:16:00] 19페이지 수집 완료 (현재 누적: 285개)
[2026-03-28 17:16:01] 20페이지 수집 완료 (현재 누적: 300개)
[2026-03-28 17:16:

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. CSV 파일 로드
file_path = 'gong-cha.csv'
df = pd.read_csv(file_path)

# 2. MySQL 연결 설정 (사용자 정보에 맞게 수정 필요)
user = 'root'      # MySQL 사용자명 (예: root)
password = '1234'  # MySQL 비밀번호
host = 'localhost'          # 호스트 (예: 127.0.0.1)
port = '3306'               # 포트 번호
db_name = 'coffee_store'    # 대상 데이터베이스

# 3. 데이터베이스 생성 (이미 존재하면 무시)
# 먼저 MySQL 서버 자체에 연결하여 DB가 없으면 생성합니다.
temp_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")
with temp_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {db_name}"))
    print(f"데이터베이스 '{db_name}'가 준비되었습니다.")

# 4. SQLAlchemy 엔진 생성 (coffee_store DB 연결)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}?charset=utf8mb4")

# 5. 데이터프레임을 MySQL 테이블로 저장
# - name: 생성할 테이블 이름 ('the_venti')
# - if_exists: 'fail' (이미 있으면 오류), 'replace' (기존 삭제 후 생성), 'append' (기존 테이블에 추가)
# - index: False (데이터프레임의 인덱스는 컬럼으로 넣지 않음)
try:
    df.to_sql(name='gong-cha', con=engine, if_exists='replace', index=False)
    print(f"테이블 'gong-cha'에 총 {len(df)}건의 데이터가 성공적으로 저장되었습니다.")
except Exception as e:
    print(f"데이터 저장 중 오류 발생: {e}")

# 연결 종료
engine.dispose()

데이터베이스 'coffee_store'가 준비되었습니다.
테이블 'gong-cha'에 총 876건의 데이터가 성공적으로 저장되었습니다.
